# Lab 1 — Material Characterization with M³ Analysis

**Physics Workspace M³ — Composite Materials for Wind Energy**

Three-scale analysis pipeline:
1. **Micro**: Fiber/matrix microstructure, void content, local properties
2. **Meso**: Layer stacking, interface behavior, ply orientation
3. **Macro**: Environment, boundaries, global response

Material: Paper mache + PVA binder + graphite coating

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../modules'))

import numpy as np
import matplotlib.pyplot as plt

from m3_analysis import M3Analysis, MacroScale, MesoScale, MicroScale, PlyLayer
from composite_model import CompositeMaterial, FabricationProcess
from mechanical_tests import (
    flexure_test, tensile_test, compression_test,
    buckling_test, impact_test, hardness_test, friction_test,
    run_all_tests
)

print('Modules loaded successfully')

In [ ]:
# ---- Step 1: Define Composite Material ----
mat = CompositeMaterial(
    fiber='waste_paper',
    matrix='pva',
    coating='graphite_coating',
    fiber_volume_fraction=0.15,
    void_fraction=0.05,
)
summary = mat.summary()
print('=== COMPOSITE MATERIAL ===')
print(f"Fiber: {summary['fiber']}")
print(f"Matrix: {summary['matrix']}")
print(f"Coating: {summary['coating']}")
print(f"Vf = {summary['Vf']:.0%}, Vv = {summary['Vv']:.0%}")
print('\nElastic constants:')
for k, v in summary['elastic'].items():
    print(f"  {k}: {v}")
print('\nStrength estimates:')
for k, v in summary['strength'].items():
    print(f"  {k}: {v}")

In [ ]:
# ---- Step 2: M³ Analysis ----
analysis = M3Analysis(material='paper_mache_pva_graphite')

# Macro: environment
analysis.macro.temperature_K = 298
analysis.macro.humidity_pct = 65
analysis.macro.wind_speed_ms = 5.0

# Meso: 3 layers
E = summary['elastic']['E1_longitudinal_GPa']
analysis.meso.add_layer(PlyLayer(
    material='graphite_coating', thickness_mm=0.5,
    elastic_modulus_GPa=8.0, density_kgm3=2000
))
analysis.meso.add_layer(PlyLayer(
    material='paper_mache_core', thickness_mm=4.0,
    elastic_modulus_GPa=E, density_kgm3=summary['elastic']['density_kgm3']
))
analysis.meso.add_layer(PlyLayer(
    material='graphite_coating', thickness_mm=0.5,
    elastic_modulus_GPa=8.0, density_kgm3=2000
))

# Micro: fiber/matrix distribution
Ef = 5.0  # waste paper fiber
Em = 2.0  # PVA matrix
analysis.micro.fiber_volume_fraction = 0.15
analysis.micro.void_fraction = 0.05
analysis.micro.fiber_diameter_um = 15.0
analysis.micro.fiber_length_mm = 3.0

result = analysis.run()

print('=== M³ ANALYSIS ===')
print('\nMacro (Environment):')
for k, v in result['macro'].items():
    print(f"  {k}: {v}")
print('\nMeso (Layers):')
for k, v in result['meso'].items():
    print(f"  {k}: {v}")
print('\nMicro (Microstructure):')
for k, v in result['micro'].items():
    print(f"  {k}: {v}")
print('\nSynthesis:')
print(f"  {result['synthesis']['overall_notes']}")

In [ ]:
# ---- Step 3: Mechanical Tests ----
E_GPa = summary['elastic']['E1_longitudinal_GPa']
strength_MPa = summary['strength']['tensile_longitudinal_MPa']

print('=== MECHANICAL TESTS ===')

bending = flexure_test(E_GPa, length_mm=100, width_mm=25, thickness_mm=5,
                       strength_MPa=strength_MPa)
print(f"\nFlexão (3-point bending):")
print(f"  Max force: {bending['max_force_N']} N")
print(f"  Strength: {bending['flexural_strength_MPa']} MPa")

tensile = tensile_test(E_GPa, width_mm=25, thickness_mm=5,
                       tensile_strength_MPa=strength_MPa)
print(f"\nTração (Tensile):")
print(f"  Max force: {tensile['max_force_N']} N")
print(f"  Strength: {tensile['tensile_strength_MPa']} MPa")

comp = compression_test(E_GPa, width_mm=25, thickness_mm=5,
                       compressive_strength_MPa=strength_MPa * 1.5)
print(f"\nCompressão:")
print(f"  Strength: {comp['compressive_strength_MPa']} MPa")

buckling = buckling_test(E_GPa, length_mm=100, width_mm=25, thickness_mm=5)
print(f"\nFlambagem (Buckling):")
print(f"  Critical load: {buckling['critical_load_N']} N")

impact = impact_test(cross_section_mm2=125)
print(f"\nChoque (Impact):")
print(f"  Toughness: {impact['toughness_kJm2']} kJ/m²")

hardness = hardness_test(E_GPa, strength_MPa)
print(f"\nDureza (Hardness):")
print(f"  Shore D: {hardness['shore_d']}")

friction = friction_test(normal_force_N=50)
print(f"\nAtrito (Friction):")
print(f"  Coeff: {friction['coefficient_friction']}")

In [ ]:
# ---- Step 4: Visualization ----
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Flexão
ax = axes[0, 0]
ss = bending['stress_strain']
ax.plot(ss['strain_pct'], ss['stress_MPa'], 'b-o', markersize=3)
ax.set_xlabel('Strain (%)'); ax.set_ylabel('Stress (MPa)')
ax.set_title('Flexão (3-point bending)')
ax.grid(True)

# Tração
ax = axes[0, 1]
ss_t = tensile['stress_strain']
ax.plot(ss_t['strain_pct'], ss_t['stress_MPa'], 'r-o', markersize=3)
ax.set_xlabel('Strain (%)'); ax.set_ylabel('Stress (MPa)')
ax.set_title('Tração (Tensile)')
ax.grid(True)

# Compressão
ax = axes[0, 2]
ss_c = comp['stress_strain']
ax.plot(ss_c['strain_pct'], ss_c['stress_MPa'], 'g-o', markersize=3)
ax.set_xlabel('Strain (%)'); ax.set_ylabel('Stress (MPa)')
ax.set_title('Compressão')
ax.grid(True)

# M³ synthesis
ax = axes[1, 0]
ax.text(0.1, 0.7, f"Macro: T={result['macro']['temperature_C']:.0f}°C, "
        f"HR={result['macro']['humidity_pct']:.0f}%", fontsize=11)
ax.text(0.1, 0.55, f"Meso: {result['meso']['n_layers']} layers, "
        f"E_eq={result['meso']['equivalent_modulus_GPa']} GPa", fontsize=11)
ax.text(0.1, 0.40, f"Micro: Vf={analysis.micro.fiber_volume_fraction:.0%}, "
        f"voids={analysis.micro.void_fraction:.1%}", fontsize=11)
ax.text(0.1, 0.25, f"Fiber spacing: {analysis.micro.fiber_spacing_um():.0f} μm", fontsize=11)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('M³ Synthesis'); ax.axis('off')

# Strength comparison
ax = axes[1, 1]
tests_names = ['Flexão', 'Tração', 'Compressão', 'Flambagem']
values = [
    bending['flexural_strength_MPa'],
    tensile['tensile_strength_MPa'],
    comp['compressive_strength_MPa'],
    buckling['critical_stress_MPa'],
]
colors_bar = ['blue', 'red', 'green', 'orange']
ax.bar(tests_names, values, color=colors_bar)
ax.set_ylabel('Stress (MPa)')
ax.set_title('Strength Comparison')
for i, v in enumerate(values):
    ax.text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=10)

# Friction force
ax = axes[1, 2]
pairs = ['composite-steel', 'composite-composite', 'composite-concrete']
frictions = [friction_test(normal_force_N=50, material_pair=p)['coefficient_friction'] for p in pairs]
ax.bar(pairs, frictions, color='purple')
ax.set_ylabel('COF'); ax.set_title('Friction Coefficient')
ax.set_xticklabels(pairs, rotation=30, ha='right')

plt.tight_layout()
plt.show()

print('=== Lab 1 Complete ===')
print(f'Material: paper_mache + PVA + graphite')
print(f'E = {E_GPa} GPa | Strength = {strength_MPa} MPa')